# Least-norm inversion: plain L² vs weighted L²(r²)

Minimal demonstration of `LinearMinimumNormInversion` for one (s, t) block
in both model-space conventions.  No prior is needed — just the forward
operator and the data.

In [ ]:
import sys, numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, 'utils')

from full_spectrum_utils import (
    RadialSpecs, build_block_forward,
    enumerate_blocks, block_data_split,
)
from normal_mode_kernel_utils import NormalModeDataRegistry, NormalModeKernelCatalog
from pygeoinf import LinearMinimumNormInversion
from pygeoinf.forward_problem import LinearForwardProblem
from pygeoinf.linear_solvers import CholeskySolver

In [ ]:
# ── Load data and kernels ────────────────────────────────────────────────────
DATA_DIR   = 'data/normal-mode-data'
KERNEL_DIR = 'data/normal-mode-kernels/kernels-all_PREM-layers_Adrian'

reg     = NormalModeDataRegistry(DATA_DIR, dataset='arwen-paula')
catalog = NormalModeKernelCatalog(KERNEL_DIR)

S_MAX = 2
blocks = enumerate_blocks(reg, S_MAX)
split  = block_data_split(reg, blocks)

# Pick the first block
block = blocks[0]
print(f'Using block s={block.s}, t={block.t}  ({len(split[block])} observations)')

In [ ]:
# ── Forward operators (one per mode) ────────────────────────────────────────
N = 50
specs_flat = RadialSpecs(n_basis=N, weighted=False)
specs_w    = RadialSpecs(n_basis=N, weighted=True)

G_flat, C_D_flat, M_flat, D = build_block_forward(
    block.s, block.t, split[block], catalog, specs_flat)

G_w, C_D_w, M_w, _ = build_block_forward(
    block.s, block.t, split[block], catalog, specs_w)

d_obs = split[block].data_vector
print(f'Data dimension: {len(d_obs)}')

In [ ]:
# ── Least-norm inversion ─────────────────────────────────────────────────────
# pygeoinf: LinearForwardProblem(G, data_error_measure=C_D)
# LinearMinimumNormInversion.minimum_norm_operator(solver) returns
# the operator that maps data → minimum-||m||_M solution fitting the data.

solver = CholeskySolver()

fp_flat = LinearForwardProblem(G_flat, data_error_measure=C_D_flat)
fp_w    = LinearForwardProblem(G_w,    data_error_measure=C_D_w)

mn_flat = LinearMinimumNormInversion(fp_flat).minimum_norm_operator(solver)
mn_w    = LinearMinimumNormInversion(fp_w   ).minimum_norm_operator(solver)

m_flat = mn_flat(d_obs)
m_w    = mn_w(d_obs)

print('Least-norm solutions computed.')

In [ ]:
# ── Plot vp profiles ──────────────────────────────────────────────────────────
R   = specs_flat.earth_radius_km
r   = np.linspace(10., R, 500)

# m_flat and m_w are nested lists matching M_st structure:
# [functions_block, euclidean_block] where
# functions_block = [vp, [vs_IC, vs_M], rho]
vp_flat = np.array([m_flat[0][0](ri) for ri in r])
vp_w    = np.array([m_w   [0][0](ri) for ri in r])

fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)

ICB = specs_flat.icb_radius_km
CMB = specs_flat.cmb_radius_km

for ax in axes:
    ax.axhspan(ICB, CMB, color='lightgray', alpha=0.4, label='outer core')
    ax.axvline(0, color='k', lw=0.6, ls=':')
    ax.set_ylabel('r (km)')
    ax.set_ylim(0, R)

# vp
axes[0].plot(vp_flat, r, color='steelblue',  lw=1.8, label='flat L²')
axes[0].plot(vp_w,    r, color='tomato', lw=1.8, ls='--', label='weighted L²(r²)')
axes[0].set_xlabel('δvp (km/s)'); axes[0].set_title('vp'); axes[0].legend(fontsize=8)

# vs
r_IC = r[r <= ICB]; r_M = r[r >= CMB]
vs_IC_flat = np.array([m_flat[0][1][0](ri) for ri in r_IC])
vs_M_flat  = np.array([m_flat[0][1][1](ri) for ri in r_M])
vs_IC_w    = np.array([m_w   [0][1][0](ri) for ri in r_IC])
vs_M_w     = np.array([m_w   [0][1][1](ri) for ri in r_M])
axes[1].plot(vs_IC_flat, r_IC, color='steelblue', lw=1.8)
axes[1].plot(vs_M_flat,  r_M,  color='steelblue', lw=1.8)
axes[1].plot(vs_IC_w,    r_IC, color='tomato', lw=1.8, ls='--')
axes[1].plot(vs_M_w,     r_M,  color='tomato', lw=1.8, ls='--')
axes[1].set_xlabel('δvs (km/s)'); axes[1].set_title('vs')

# rho
rho_flat = np.array([m_flat[0][2](ri) for ri in r])
rho_w    = np.array([m_w   [0][2](ri) for ri in r])
axes[2].plot(rho_flat, r, color='steelblue', lw=1.8, label='flat L²')
axes[2].plot(rho_w,    r, color='tomato', lw=1.8, ls='--', label='weighted L²(r²)')
axes[2].set_xlabel('δρ (g/cm³)'); axes[2].set_title('rho'); axes[2].legend(fontsize=8)

plt.suptitle(f'Least-norm solution  s={block.s}, t={block.t}', fontsize=11)
plt.tight_layout()
plt.show()

print(f'||m_flat||  = {M_flat.norm(m_flat):.4e}')
print(f'||m_w||_w   = {M_w.norm(m_w):.4e}')

In [ ]:
# ── Data fit comparison ───────────────────────────────────────────────────────
d_pred_flat = G_flat(m_flat)
d_pred_w    = G_w(m_w)
err = split[block].error_vector

idx = np.arange(len(d_obs))
fig, ax = plt.subplots(figsize=(12, 3))
ax.errorbar(idx,       d_obs,       yerr=err, fmt='o', ms=4, color='k',
            ecolor='k', elinewidth=0.8, capsize=2, label='observed', zorder=3)
ax.plot(idx, d_pred_flat, 's', ms=5, color='steelblue', label='flat L²', zorder=4)
ax.plot(idx, d_pred_w,    '^', ms=5, color='tomato',    label='weighted L²(r²)', zorder=4)
ax.axhline(0, color='gray', lw=0.5)
ax.set_xlabel('observation index'); ax.set_ylabel('splitting coefficient')
ax.set_title('Data fit'); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

chi_flat = float(np.sqrt(np.mean(((d_obs - d_pred_flat)/err)**2)))
chi_w    = float(np.sqrt(np.mean(((d_obs - d_pred_w   )/err)**2)))
print(f'RMS χ: flat={chi_flat:.3f}  weighted={chi_w:.3f}  (target ≈ 1)')